# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/faja27/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding A — "What Predicts Health?" (ML Appendix, Random Forest)

The paper reports Average Position (43%) and Impressions (32%) as the top two Random Forest predictors of `health_score`, and is admirably upfront that the appendix flags this as descriptive, not causal, because the target is partly built from the same inputs.

**My methodology question:** `Health Score = Impressions(30) + Position(30) + CTR(20) + Scroll Depth(20)` — by construction, 60% of the score's formula weight already sits on position + impressions. Given that, is a feature-importance chart here telling us something new, or mostly re-deriving the known formula weights? (Code cell below checks how close the RF's 75% combined importance sits to the formula's 60% structural share — they're in the same neighborhood, which is exactly what you'd expect from a target partly built from its own predictors, not necessarily evidence of anything beyond the formula.) A version of this appendix that predicted an *independent downstream outcome* (e.g. next-quarter growth) from position/impressions would carry the claim better than predicting a score that already contains them as line items.

### Finding B — "What Predicts Growth?" (ML Appendix, Logistic Regression, 71% holdout accuracy)

The growing/declining label comes from a **30-day-vs-previous-30-day impression change** (>10% up = growing, >10% down = declining, per the paper's own Finding Tags section) — a real, disclosed, non-circular label definition. Good.

**My methodology question:** the Methodology page discloses "Logistic Regression (80/20 split)" for 57 brands, but doesn't say whether that split was **grouped by brand**. That's exactly the gap I had to close in my own
Week-5 model (my baseline candidates span only 29 clients, and a plain random split let some of a client's own pages sit in both train and test). If the paper's 80/20 split was page-level random across 57 brands, the 71%
holdout accuracy may include some of that same brand-memorization effect — worth a footnote either way, since readers can't tell from the current disclosure whether "unseen page" also means "unseen brand." Section 2 below
shows how large that gap turned out to be for my own model, as a concrete before/after.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


# ML-09 doesn't need new data pulls for this section - just a code-backed recap of the two
# findings' disclosed methodology details, so the audit above is traceable to real numbers.

health_score_weights = {"impressions": 30, "position": 30, "ctr": 20, "scroll_depth": 20}
rf_importance_reported = {"avg_position": 43, "impressions": 32, "scroll_depth": 15, "ctr": 8}

structural_share_pos_impr = (health_score_weights["position"] + health_score_weights["impressions"]) / 100
rf_share_pos_impr = (rf_importance_reported["avg_position"] + rf_importance_reported["impressions"]) / 100

print("Finding A - structural check")
print(f"  formula weight on position+impressions:      {structural_share_pos_impr:.0%}")
print(f"  RF combined importance on position+impressions: {rf_share_pos_impr:.0%}")
print(f"  gap: {rf_share_pos_impr - structural_share_pos_impr:+.0%} "
      f"(same neighborhood - consistent with 'descriptive, not causal', as the paper itself says)")

growth_label_rule = "30d-vs-prev-30d impression change: >10% = growing, >10% = declining (paper's own Finding Tags)"
disclosed_split = "Logistic Regression (80/20 split) - Methodology page; grouping by brand not stated"

print("\nFinding B - disclosed facts")
print(f"  label rule:      {growth_label_rule}")
print(f"  split disclosed: {disclosed_split}")
print("  -> not a circularity problem (label is a real trend, not a sibling of a feature),")
print("     but the split-grouping gap is untestable from outside - see section 2 for my own version of it.")

Finding A - structural check
  formula weight on position+impressions:      60%
  RF combined importance on position+impressions: 75%
  gap: +15% (same neighborhood - consistent with 'descriptive, not causal', as the paper itself says)

Finding B - disclosed facts
  label rule:      30d-vs-prev-30d impression change: >10% = growing, >10% = declining (paper's own Finding Tags)
  split disclosed: Logistic Regression (80/20 split) - Methodology page; grouping by brand not stated
  -> not a circularity problem (label is a real trend, not a sibling of a feature),
     but the split-grouping gap is untestable from outside - see section 2 for my own version of it.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*


**Before:** a plain random 80/20 split (stratified on the label, `client_id` ignored) — the kind of split the paper's Methodology page doesn't rule out for its own 80/20 splits.

**After:** the `GroupShuffleSplit` by `client_id` I actually used in Week 5 — verified zero client overlap between train and test.

Same features, same model (random forest), same label, same K values — only the split changes.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/faja27/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import roc_auc_score

SEED = 42
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# --- reproduce week-4/5 lane, label, and features exactly ---
in_range = df["position_tier"].isin(["page_1", "striking"])
has_demand = df["impression_tier"].isin(["moderate", "good", "excellent"])
is_candidate = in_range & has_demand
df["quick_win_score"] = np.where(is_candidate, df["impressions_90d"], 0)
cand = df[is_candidate].copy().reset_index(drop=True)

bench_pool = df[df["position_tier"].isin(["page_1", "striking"])]
benchmark_ctr = 100 * bench_pool["clicks_90d"].sum() / bench_pool["impressions_90d"].sum()
cand["label"] = (cand["ctr"] < benchmark_ctr).astype(int)
cand["has_keyword_data"] = cand["search_volume"].notna().astype(int)

num_feats = ["search_volume", "competition", "cpc", "word_count", "char_count",
             "content_age_days", "days_since_last_update", "impressions_90d",
             "engagement_rate", "scroll_rate", "ai_traffic_pct", "avg_position", "has_keyword_data"]
cat_feats = ["competition_level", "content_type", "main_intent", "age_tier", "freshness_tier",
             "position_tier", "impression_tier"]

X = cand[num_feats + cat_feats].copy()
for c in cat_feats:
    X[c] = X[c].fillna("unknown")
y = cand["label"].values
groups = cand["client_id"].values

pre = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), num_feats),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_feats),
])

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

def fit_and_score(train_idx, test_idx, label_name):
    rf = Pipeline([("pre", pre), ("clf", RandomForestClassifier(
        n_estimators=300, max_depth=6, min_samples_leaf=20, random_state=SEED, n_jobs=-1))])
    rf.fit(X.iloc[train_idx], y[train_idx])
    proba = rf.predict_proba(X.iloc[test_idx])[:, 1]
    auc = roc_auc_score(y[test_idx], proba)
    row = {"split": label_name, "test_base_rate": round(y[test_idx].mean(), 3), "auc": round(auc, 3)}
    for k in (50, 100, 200):
        row[f"P@{k}"] = round(precision_at_k(proba, y[test_idx], k), 3)
    return row

# BEFORE: naive random split, client_id ignored
before_train, before_test = train_test_split(np.arange(len(X)), test_size=0.2, random_state=SEED, stratify=y)
before = fit_and_score(before_train, before_test, "BEFORE: random split (client_id ignored)")

# AFTER: grouped split by client_id (what I actually used in week 5)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
after_train, after_test = next(gss.split(X, y, groups))
overlap = set(cand.iloc[after_train]["client_id"]) & set(cand.iloc[after_test]["client_id"])
after = fit_and_score(after_train, after_test, "AFTER: grouped split by client_id")

comparison = pd.DataFrame([before, after])
print(comparison.to_string(index=False))
print(f"\nclient overlap in the AFTER split (must be empty): {overlap}")
print(f"AUC gap (before - after): {before['auc'] - after['auc']:+.3f} — a modest but real inflation from")
print("ignoring client grouping. precision@K is close either way, which is reassuring: this model isn't")
print("leaning heavily on memorizing which client a page belongs to.")


                                   split  test_base_rate   auc  P@50  P@100  P@200
BEFORE: random split (client_id ignored)           0.698 0.758  0.92   0.95  0.935
       AFTER: grouped split by client_id           0.681 0.698  0.94   0.89  0.915

client overlap in the AFTER split (must be empty): set()
AUC gap (before - after): +0.060 — a modest but real inflation from
ignoring client grouping. precision@K is close either way, which is reassuring: this model isn't
leaning heavily on memorizing which client a page belongs to.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Running the attack checklist against the exact feature set trained in Week 5 (`num_feats` + `cat_feats` above), plus the one item the checklist insists on doing, not just checking: deliberately re-introduce a leaky feature and confirm the harness actually notices.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

forbidden = {"ctr", "clicks_90d", "trend_direction", "trend_pct", "health_score", "priority_score",
             "action_type", "needs_ctr_fix", "is_quick_win", "quick_win_score", "reason_code", "action"}
feature_set = set(num_feats + cat_feats)

print("--- attack checklist ---")
print(f"[{'x' if not forbidden & feature_set else ' '}] no label-derived or sibling columns in features "
      f"(overlap: {forbidden & feature_set or 'none'})")
print(f"[x] no product flags as features (none of FlyRank's own flags exist in this anonymized dataset at all)")
print("[x] population selection disclosed: modeling only runs on the candidate pool "
      "(position_tier in page_1/striking, impression_tier moderate+) - that IS my lane, stated in week 4/5, "
      "not hidden here")
print("[x] split grouped by client_id, verified zero overlap (section 2)")
print("[x] base rate printed next to every metric (section 2 table)")
print("[x] top feature importance sanity-checked in week 5: engagement_rate led at 0.006 permutation "
      "importance - low and plausible, not a 'too good' single-feature spike")
print("[x] metrics computed out-of-fold on a held-out test split, never in-sample")

# the checklist's own verification step: inject a known leaky feature and confirm the harness reacts
X_leaky = X.copy()
X_leaky["ctr_DELIBERATE_LEAK"] = cand["ctr"].values
pre_leaky = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), num_feats + ["ctr_DELIBERATE_LEAK"]),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_feats),
])
rf_leaky = Pipeline([("pre", pre_leaky), ("clf", RandomForestClassifier(
    n_estimators=300, max_depth=6, min_samples_leaf=20, random_state=SEED, n_jobs=-1))])
rf_leaky.fit(X_leaky.iloc[after_train], y[after_train])
proba_leaky = rf_leaky.predict_proba(X_leaky.iloc[after_test])[:, 1]
auc_leaky = roc_auc_score(y[after_test], proba_leaky)

print(f"\n--- harness verification ---")
print(f"honest AUC (no leak):              {after['auc']:.3f}")
print(f"AUC WITH ctr deliberately injected: {auc_leaky:.3f}")
assert auc_leaky > 0.98, "harness did not react to an obvious leak - the test itself would be broken"
print("harness correctly jumps to ~1.0 when fed the label's own source column - the honest 0.698 earlier")
print("is a real number, not a harness that simply can't detect leakage.")

--- attack checklist ---
[x] no label-derived or sibling columns in features (overlap: none)
[x] no product flags as features (none of FlyRank's own flags exist in this anonymized dataset at all)
[x] population selection disclosed: modeling only runs on the candidate pool (position_tier in page_1/striking, impression_tier moderate+) - that IS my lane, stated in week 4/5, not hidden here
[x] split grouped by client_id, verified zero overlap (section 2)
[x] base rate printed next to every metric (section 2 table)
[x] top feature importance sanity-checked in week 5: engagement_rate led at 0.006 permutation importance - low and plausible, not a 'too good' single-feature spike
[x] metrics computed out-of-fold on a held-out test split, never in-sample

--- harness verification ---
honest AUC (no leak):              0.698
AUC WITH ctr deliberately injected: 1.000
harness correctly jumps to ~1.0 when fed the label's own source column - the honest 0.698 earlier
is a real number, not a harness t

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*


**Original (Week-5 notebook, section 4):**
- "Both models roughly triple the baseline's precision@K... That's the real finding: volume tells you how big
- an opportunity is, not whether it's actually broken."

That's stronger than the evidence supports — "the real finding" and "tells you" read as settled fact from one
sample and one split.

**Rewritten:**
- In this sample, under a grouped holdout split, logistic regression and random forest showed higher
- precision@K (roughly 0.87-0.94) than the volume-only baseline (roughly 0.66-0.68, close to the label's own
- base rate). That's a directional, decision-support signal — not a guaranteed result — that content and
- engagement features carry information about efficiency the volume-only rule doesn't capture on its own.
- Worth validating on more clients before it drives a live triage queue.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# no computation needed to rewrite a sentence - this cell documents the before/after pair
# programmatically so the rewrite is easy to diff against the original in review.

original_claim = (
    "Both models roughly triple the baseline's precision@K... That's the real finding: volume tells you "
    "how big an opportunity is, not whether it's actually broken."
)
safe_claim = (
    "In this sample, under a grouped holdout split, logistic regression and random forest showed higher "
    "precision@K (roughly 0.87-0.94) than the volume-only baseline (roughly 0.66-0.68, close to the label's "
    "own base rate). That's a directional, decision-support signal - not a guaranteed result - that content "
    "and engagement features carry information about efficiency the volume-only rule doesn't capture on its "
    "own. Worth validating on more clients before it drives a live triage queue."
)

safe_words = ["observed", "measured", "directional", "decision-support", "sample", "this sample", "signal"]
# checked as standalone claims, not negations - "not a guaranteed result" doesn't count as claiming one
overclaim_words = ["real finding", "tells you", "this proves", "always true", "guarantees that"]

print("ORIGINAL:\n", original_claim)
print("\nREWRITTEN:\n", safe_claim)
print("\noverclaim language remaining in rewrite:",
      [w for w in overclaim_words if w in safe_claim.lower()] or "none")
print("safe-language markers present in rewrite:",
      [w for w in safe_words if w in safe_claim.lower()])


ORIGINAL:
 Both models roughly triple the baseline's precision@K... That's the real finding: volume tells you how big an opportunity is, not whether it's actually broken.

REWRITTEN:
 In this sample, under a grouped holdout split, logistic regression and random forest showed higher precision@K (roughly 0.87-0.94) than the volume-only baseline (roughly 0.66-0.68, close to the label's own base rate). That's a directional, decision-support signal - not a guaranteed result - that content and engagement features carry information about efficiency the volume-only rule doesn't capture on its own. Worth validating on more clients before it drives a live triage queue.

overclaim language remaining in rewrite: none
safe-language markers present in rewrite: ['directional', 'decision-support', 'sample', 'this sample', 'signal']


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.